# Iris Classifier - MLOps Pipeline

Notebook de prueba para la plataforma MLOps.
Entrena un clasificador Random Forest sobre el dataset Iris.

In [ ]:
# Papermill injected parameters (defaults for local testing)
MODEL_OUTPUT_PATH = "model.joblib"
PIPELINE_ID = "local-test"
MLFLOW_TRACKING_URI = "http://mlflow:5000"
MLFLOW_RUN_ID = ""

In [ ]:
MODEL_NAME = "iris-classifier"
VERSION = "1"

# Hyperparameters
N_ESTIMATORS = 100
MAX_DEPTH = 5
TEST_SIZE = 0.2
RANDOM_STATE = 42

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
import numpy as np

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split into train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Test samples: {len(X_test)}")
print(f"Features: {iris.feature_names}")
print(f"Classes: {list(iris.target_names)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
import mlflow

# Set up MLflow
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Use the run created by the pipeline (or create a new one for local testing)
if MLFLOW_RUN_ID:
    run = mlflow.start_run(run_id=MLFLOW_RUN_ID)
else:
    run = mlflow.start_run(run_name=f"{MODEL_NAME}-local")

# Log hyperparameters
mlflow.log_param("n_estimators", N_ESTIMATORS)
mlflow.log_param("max_depth", MAX_DEPTH)
mlflow.log_param("test_size", TEST_SIZE)

# Train model
model = RandomForestClassifier(
    n_estimators=N_ESTIMATORS,
    max_depth=MAX_DEPTH,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)

# Evaluate
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average="weighted")

# Log metrics to MLflow
mlflow.log_metric("accuracy", accuracy)
mlflow.log_metric("f1_score", f1)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"\nClassification Report:\n{classification_report(y_test, y_pred, target_names=iris.target_names)}")

mlflow.end_run()

In [ ]:
import joblib

# Save model to the path expected by the pipeline
joblib.dump(model, MODEL_OUTPUT_PATH)
print(f"Model saved to: {MODEL_OUTPUT_PATH}")
print(f"Model: {MODEL_NAME} v{VERSION}")
print(f"Pipeline ID: {PIPELINE_ID}")